In [ ]:
!pip install requests pandas numpy scikit-learn sentence-transformers tqdm matplotlib seaborn --quiet

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import time
import random
import os
import pickle
from tqdm import tqdm
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Create output directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/embeddings', exist_ok=True)

print('✅ All imports successful!')

## 🌐 Step 2 — Scrape LeetCode Problem Metadata

LeetCode exposes a **public GraphQL API** that returns problem metadata without needing authentication. We'll use it to pull real problem titles, difficulty, tags, acceptance rates, etc.

In [ ]:
# ── LeetCode Public GraphQL API ──────────────────────────────────────────────

LEETCODE_API = "https://leetcode.com/graphql"

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://leetcode.com/problemset/"
}

# GraphQL query to fetch all problems with metadata
PROBLEMS_QUERY = """
query problemsetQuestionList($categorySlug: String, $limit: Int, $skip: Int, $filters: QuestionListFilterInput) {
  problemsetQuestionList: questionList(
    categorySlug: $categorySlug
    limit: $limit
    skip: $skip
    filters: $filters
  ) {
    total: totalNum
    questions: data {
      questionId
      questionFrontendId
      title
      titleSlug
      difficulty
      acRate
      topicTags {
        name
        slug
      }
      isPaidOnly
      status
      freqBar
    }
  }
}
"""

def fetch_problems_batch(skip=0, limit=100):
    """Fetch a batch of problems from LeetCode GraphQL API."""
    payload = {
        "query": PROBLEMS_QUERY,
        "variables": {
            "categorySlug": "",
            "limit": limit,
            "skip": skip,
            "filters": {}
        }
    }
    try:
        resp = requests.post(LEETCODE_API, json=payload, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        return data['data']['problemsetQuestionList']
    except Exception as e:
        print(f"  ⚠️  Error at skip={skip}: {e}")
        return None

print('🔍 Fetching first batch to check total count...')
first = fetch_problems_batch(skip=0, limit=1)
if first:
    total = first['total']
    print(f'✅ Total problems available: {total}')
else:
    print('❌ Could not reach API. Will use fallback synthetic data in Step 2b.')
    total = 0

In [ ]:
# ── Scrape all free problems (skip paid-only) ─────────────────────────────────
# Fetches up to 2500 problems in batches of 100

all_problems = []
BATCH_SIZE   = 100
MAX_PROBLEMS = min(total, 2500)  # cap at 2500 for Colab memory

if total > 0:
    print(f'\n🚀 Scraping {MAX_PROBLEMS} problems in batches of {BATCH_SIZE}...')
    for skip in tqdm(range(0, MAX_PROBLEMS, BATCH_SIZE)):
        result = fetch_problems_batch(skip=skip, limit=BATCH_SIZE)
        if result and result.get('questions'):
            all_problems.extend(result['questions'])
        time.sleep(0.5)  # be polite to the server

    print(f'\n✅ Scraped {len(all_problems)} problems total')
else:
    print('⚠️  Skipping scrape — will generate synthetic problems in next cell.')

In [ ]:
# ── 2b: Fallback — Synthetic Problem Dataset ──────────────────────────────────
# Run this cell only if the scraper failed OR you want to add extra synthetic problems

TOPICS = [
    'Array', 'String', 'Hash Table', 'Dynamic Programming', 'Math',
    'Sorting', 'Greedy', 'Depth-First Search', 'Breadth-First Search',
    'Binary Search', 'Tree', 'Graph', 'Backtracking', 'Stack', 'Heap',
    'Linked List', 'Sliding Window', 'Two Pointers', 'Trie', 'Union Find'
]

DIFFICULTY_DIST = ['Easy'] * 30 + ['Medium'] * 50 + ['Hard'] * 20

def generate_synthetic_problems(n=500, start_id=3000):
    problems = []
    for i in range(n):
        pid    = start_id + i
        diff   = random.choice(DIFFICULTY_DIST)
        n_tags = random.randint(1, 4)
        tags   = random.sample(TOPICS, n_tags)

        ac_base = {'Easy': 0.65, 'Medium': 0.45, 'Hard': 0.30}[diff]
        ac_rate = round(np.clip(np.random.normal(ac_base, 0.1), 0.05, 0.95), 3)

        problems.append({
            'questionId':         str(pid),
            'questionFrontendId': str(pid),
            'title':              f'Synthetic Problem {pid}',
            'titleSlug':          f'synthetic-problem-{pid}',
            'difficulty':         diff,
            'acRate':             ac_rate,
            'topicTags':          [{'name': t, 'slug': t.lower().replace(' ', '-')} for t in tags],
            'isPaidOnly':         False,
            'freqBar':            round(random.uniform(0, 100), 1)
        })
    return problems

# If scraping failed OR we have < 500 problems, pad with synthetic
if len(all_problems) < 500:
    synthetic = generate_synthetic_problems(n=500)
    all_problems.extend(synthetic)
    print(f'✅ Added 500 synthetic problems. Total: {len(all_problems)}')
else:
    print(f'✅ Using {len(all_problems)} scraped problems — no synthetic padding needed')

## 🧹 Step 3 — Clean & Structure the Problems DataFrame

In [ ]:
def parse_problems(raw_list):
    rows = []
    for p in raw_list:
        # Skip paid-only problems (no public data)
        if p.get('isPaidOnly', False):
            continue

        tags = [t['name'] for t in p.get('topicTags', [])]

        rows.append({
            'problem_id':   int(p['questionFrontendId']),
            'title':        p['title'],
            'slug':         p['titleSlug'],
            'difficulty':   p['difficulty'],      # Easy / Medium / Hard
            'acceptance':   round(float(p.get('acRate', 0.5)), 4),
            'frequency':    float(p.get('freqBar') or 0.0),
            'topics':       tags,                 # list of strings
            'topic_str':    ' '.join(tags),       # for TF-IDF / embeddings
            'n_topics':     len(tags),
        })
    return pd.DataFrame(rows)

problems_df = parse_problems(all_problems)

# Remove duplicates (same problem_id)
problems_df = problems_df.drop_duplicates('problem_id').reset_index(drop=True)

# Add numeric difficulty
diff_map = {'Easy': 1, 'Medium': 2, 'Hard': 3}
problems_df['difficulty_num'] = problems_df['difficulty'].map(diff_map)

# Save raw
problems_df.to_csv('data/raw/problems_raw.csv', index=False)
print(f'✅ Problems DataFrame: {problems_df.shape}')
problems_df.head()

In [ ]:
# ── EDA: Distributions ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Problem Dataset Overview', fontsize=14, fontweight='bold')

# Difficulty distribution
diff_counts = problems_df['difficulty'].value_counts()
axes[0].pie(diff_counts, labels=diff_counts.index,
            colors=['#4ade80','#facc15','#f87171'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Difficulty Distribution')

# Acceptance rate
axes[1].hist(problems_df['acceptance'], bins=30, color='#818cf8', edgecolor='white')
axes[1].set_title('Acceptance Rate Distribution')
axes[1].set_xlabel('Acceptance Rate')

# Top 10 topics
from collections import Counter
topic_counts = Counter(t for tags in problems_df['topics'] for t in tags)
top10 = pd.Series(dict(topic_counts.most_common(10)))
axes[2].barh(top10.index[::-1], top10.values[::-1], color='#38bdf8')
axes[2].set_title('Top 10 Topics')

plt.tight_layout()
plt.savefig('data/raw/eda_problems.png', dpi=120)
plt.show()
print(f'Total unique topics: {len(topic_counts)}')

## 👤 Step 4 — Generate Synthetic User Interaction Data

Since LeetCode user solve histories are private, we simulate realistic interactions using behavioral profiles.

In [ ]:
# ── User Profile Archetypes ───────────────────────────────────────────────────
# Each archetype has a preferred difficulty and topic affinity

USER_ARCHETYPES = {
    'beginner': {
        'diff_weights': [0.70, 0.25, 0.05],   # Easy / Medium / Hard
        'n_solved_range': (20, 80),
        'fav_topics': ['Array', 'String', 'Math', 'Hash Table']
    },
    'intermediate': {
        'diff_weights': [0.25, 0.55, 0.20],
        'n_solved_range': (100, 350),
        'fav_topics': ['Dynamic Programming', 'Tree', 'Binary Search', 'Sliding Window']
    },
    'advanced': {
        'diff_weights': [0.10, 0.40, 0.50],
        'n_solved_range': (300, 800),
        'fav_topics': ['Graph', 'Backtracking', 'Trie', 'Union Find', 'Heap']
    },
    'contest': {
        'diff_weights': [0.05, 0.35, 0.60],
        'n_solved_range': (500, 1200),
        'fav_topics': ['Dynamic Programming', 'Math', 'Graph', 'Greedy']
    }
}

N_USERS = 2000   # synthetic users

problems_by_diff = {
    d: problems_df[problems_df['difficulty'] == d]['problem_id'].tolist()
    for d in ['Easy', 'Medium', 'Hard']
}

def get_problems_for_user(archetype_name, n_solve):
    arch   = USER_ARCHETYPES[archetype_name]
    diffs  = ['Easy', 'Medium', 'Hard']
    chosen = []

    # Weight by favorite topics — pick those first
    fav_topics = arch['fav_topics']
    fav_mask   = problems_df['topics'].apply(lambda ts: any(t in fav_topics for t in ts))
    fav_ids    = problems_df[fav_mask]['problem_id'].tolist()

    # Sample across difficulties
    for d, w in zip(diffs, arch['diff_weights']):
        pool = problems_by_diff[d]
        n    = max(1, int(n_solve * w))
        fav_pool = [pid for pid in fav_ids if pid in pool]
        n_fav    = min(len(fav_pool), int(n * 0.6))
        n_rest   = n - n_fav
        rest_pool = [pid for pid in pool if pid not in fav_pool]
        chosen.extend(random.sample(fav_pool, n_fav) if fav_pool else [])
        chosen.extend(random.sample(rest_pool, min(n_rest, len(rest_pool))))

    return list(set(chosen))

print('✅ Archetype helper ready')

In [ ]:
# ── Generate Interactions ─────────────────────────────────────────────────────

interactions = []
users_meta   = []
archetype_names = list(USER_ARCHETYPES.keys())

for uid in tqdm(range(N_USERS), desc='Generating users'):
    arch_name = random.choice(archetype_names)
    arch      = USER_ARCHETYPES[arch_name]
    n_solve   = random.randint(*arch['n_solved_range'])
    n_solve   = min(n_solve, len(problems_df))  # can't solve more than exist

    solved_ids = get_problems_for_user(arch_name, n_solve)

    # Assign signup date in the past 2 years
    signup_days_ago = random.randint(30, 730)
    signup_date     = datetime.now() - timedelta(days=signup_days_ago)

    users_meta.append({
        'user_id':   uid,
        'archetype': arch_name,
        'n_solved':  len(solved_ids),
        'signup_date': signup_date.strftime('%Y-%m-%d')
    })

    for pid in solved_ids:
        prob_row = problems_df[problems_df['problem_id'] == pid].iloc[0]

        # Time to solve (seconds): harder → more time, noisy
        base_time = {'Easy': 600, 'Medium': 1800, 'Hard': 3600}[prob_row['difficulty']]
        time_taken = max(60, int(np.random.normal(base_time, base_time * 0.3)))

        # Number of attempts
        attempts = max(1, int(np.random.exponential(1.5)))

        # Implicit rating: 1.0=solved, 0.5=attempted, 0=skipped
        rating = 1.0 if attempts <= 3 else 0.7

        # Solve date: random day after signup
        days_after = random.randint(0, signup_days_ago)
        solve_date = signup_date + timedelta(days=days_after)

        interactions.append({
            'user_id':    uid,
            'problem_id': pid,
            'rating':     rating,
            'time_taken': time_taken,
            'attempts':   attempts,
            'solved_at':  solve_date.strftime('%Y-%m-%d')
        })

interactions_df = pd.DataFrame(interactions)
users_df        = pd.DataFrame(users_meta)

print(f'\n✅ Users: {users_df.shape}')
print(f'✅ Interactions: {interactions_df.shape}')
print(f'✅ Avg solves/user: {interactions_df.groupby("user_id").size().mean():.1f}')

## ⚙️ Step 5 — Feature Engineering

In [ ]:
# ── 5a: User-level Features ───────────────────────────────────────────────────

TOPICS_LIST = [
    'Array','String','Hash Table','Dynamic Programming','Math',
    'Sorting','Greedy','Depth-First Search','Breadth-First Search',
    'Binary Search','Tree','Graph','Backtracking','Stack','Heap',
    'Linked List','Sliding Window','Two Pointers','Trie','Union Find'
]

# Merge interactions with problem metadata
merged = interactions_df.merge(problems_df[['problem_id','difficulty_num','topics','acceptance']],
                               on='problem_id', how='left')

def compute_user_features(user_group):
    """Compute a feature row for one user."""
    uid    = user_group['user_id'].iloc[0]
    n_sol  = len(user_group)
    avg_t  = user_group['time_taken'].mean()
    avg_att= user_group['attempts'].mean()

    diff_dist = user_group['difficulty_num'].value_counts(normalize=True).to_dict()
    pct_easy  = diff_dist.get(1, 0.0)
    pct_med   = diff_dist.get(2, 0.0)
    pct_hard  = diff_dist.get(3, 0.0)

    # Per-topic mastery vector
    topic_mastery = {}
    for topic in TOPICS_LIST:
        mask   = user_group['topics'].apply(lambda ts: topic in (ts if isinstance(ts, list) else []))
        subset = user_group[mask]
        if len(subset) == 0:
            topic_mastery[f'skill_{topic}'] = 0.0
        else:
            avg_rating = subset['rating'].mean()
            speed_norm = 1 / np.log1p(subset['time_taken'].mean())
            topic_mastery[f'skill_{topic}'] = round(avg_rating * speed_norm * 10, 4)

    return {
        'user_id':        uid,
        'n_solved':       n_sol,
        'avg_time_sec':   round(avg_t, 1),
        'avg_attempts':   round(avg_att, 2),
        'pct_easy':       round(pct_easy, 3),
        'pct_medium':     round(pct_med, 3),
        'pct_hard':       round(pct_hard, 3),
        **topic_mastery
    }

print('⏳ Computing user features (this takes ~1-2 min)...')
user_features_rows = []
for uid, grp in tqdm(merged.groupby('user_id')):
    user_features_rows.append(compute_user_features(grp))

user_features_df = pd.DataFrame(user_features_rows)

# Normalize skill columns
skill_cols = [c for c in user_features_df.columns if c.startswith('skill_')]
scaler = MinMaxScaler()
user_features_df[skill_cols] = scaler.fit_transform(user_features_df[skill_cols])

print(f'\n✅ User features shape: {user_features_df.shape}')
user_features_df.head(3)

In [ ]:
# ── 5b: Problem Features ──────────────────────────────────────────────────────

# One-hot encode topics
topic_onehot = pd.DataFrame(0, index=problems_df.index, columns=TOPICS_LIST)
for i, row in problems_df.iterrows():
    for t in row['topics']:
        if t in TOPICS_LIST:
            topic_onehot.at[i, t] = 1

# Rename topic cols to avoid clash
topic_onehot.columns = [f'topic_{c}' for c in topic_onehot.columns]

problem_features_df = pd.concat([
    problems_df[['problem_id','difficulty_num','acceptance','frequency','n_topics']],
    topic_onehot
], axis=1)

# Normalize numeric columns
num_cols = ['acceptance','frequency','n_topics']
problem_features_df[num_cols] = MinMaxScaler().fit_transform(problem_features_df[num_cols])

print(f'✅ Problem features shape: {problem_features_df.shape}')
problem_features_df.head(3)

In [ ]:
# ── 5c: Interaction Matrix (for CF model) ─────────────────────────────────────

# Re-index user_id and problem_id to contiguous integers
le_user = LabelEncoder()
le_prob = LabelEncoder()

interactions_df['user_idx']    = le_user.fit_transform(interactions_df['user_id'])
interactions_df['problem_idx'] = le_prob.fit_transform(interactions_df['problem_id'])

N_USERS_ENC    = interactions_df['user_idx'].nunique()
N_PROBLEMS_ENC = interactions_df['problem_idx'].nunique()

print(f'✅ Encoded users:    {N_USERS_ENC}')
print(f'✅ Encoded problems: {N_PROBLEMS_ENC}')
print(f'✅ Sparsity: {1 - len(interactions_df)/(N_USERS_ENC*N_PROBLEMS_ENC):.4%}')

# Train / validation / test split (80 / 10 / 10)
train_df, temp_df = train_test_split(interactions_df, test_size=0.2, random_state=SEED)
val_df,   test_df = train_test_split(temp_df,          test_size=0.5, random_state=SEED)

print(f'\nSplit sizes → Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 🔤 Step 6 — Problem Embeddings (Sentence Transformer)

In [ ]:
from sentence_transformers import SentenceTransformer

# Build descriptive text per problem for embedding
def build_problem_text(row):
    topics_str = ', '.join(row['topics']) if row['topics'] else 'general'
    return (
        f"{row['title']}. "
        f"Difficulty: {row['difficulty']}. "
        f"Topics: {topics_str}. "
        f"Acceptance rate: {row['acceptance']:.0%}."
    )

problems_df['embed_text'] = problems_df.apply(build_problem_text, axis=1)
print('Sample embed text:')
print(problems_df['embed_text'].iloc[0])

print('\n⏳ Loading sentence transformer (downloads ~80MB once)...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('⏳ Encoding problems (may take 2-4 min on Colab)...')
problem_embeddings = embedder.encode(
    problems_df['embed_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True   # cosine similarity ready
)

print(f'\n✅ Embeddings shape: {problem_embeddings.shape}')  # (n_problems, 384)

In [ ]:
# Quick sanity check: nearest neighbors for a sample problem
from sklearn.metrics.pairwise import cosine_similarity

SAMPLE_IDX = 0
sample_title = problems_df.iloc[SAMPLE_IDX]['title']
sims = cosine_similarity(problem_embeddings[SAMPLE_IDX:SAMPLE_IDX+1], problem_embeddings)[0]
top5 = np.argsort(sims)[::-1][1:6]

print(f'\n🔍 Problems most similar to: "{sample_title}"')
for idx in top5:
    row = problems_df.iloc[idx]
    print(f'  [{sims[idx]:.3f}] {row["title"]} ({row["difficulty"]}) — {row["topic_str"][:50]}')

## 💾 Step 7 — Save All Processed Data

In [ ]:
# ── Save all artifacts ────────────────────────────────────────────────────────

# 1. Raw CSVs
problems_df.to_csv('data/raw/problems.csv', index=False)
users_df.to_csv('data/raw/users.csv', index=False)
interactions_df.to_csv('data/raw/interactions.csv', index=False)

# 2. Processed feature matrices
user_features_df.to_csv('data/processed/user_features.csv', index=False)
problem_features_df.to_csv('data/processed/problem_features.csv', index=False)

# 3. Train / val / test splits
train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

# 4. Embeddings
np.save('data/embeddings/problem_embeddings.npy', problem_embeddings)

# 5. Encoders & scaler (needed at inference time)
with open('data/processed/encoders.pkl', 'wb') as f:
    pickle.dump({'le_user': le_user, 'le_prob': le_prob, 'scaler': scaler}, f)

# 6. Metadata
meta = {
    'n_users':    N_USERS_ENC,
    'n_problems': N_PROBLEMS_ENC,
    'n_topics':   len(TOPICS_LIST),
    'topics':     TOPICS_LIST,
    'embed_dim':  problem_embeddings.shape[1]
}
with open('data/processed/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ All data saved!')
print('\n📁 Directory structure:')
for root, dirs, files in os.walk('data'):
    level = root.replace('data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f'{indent}  {file}  ({size/1024:.1f} KB)')

In [ ]:
# ── Optional: Mount Drive & backup ───────────────────────────────────────────
# Uncomment to save to your Google Drive so data persists between sessions

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('data', '/content/drive/MyDrive/leetcode_recommender/data',
#                 dirs_exist_ok=True)
# print('✅ Backed up to Google Drive!')

## ✅ Phase 1 Complete — Summary

| Artifact | Location | Description |
|---|---|---|
| `problems.csv` | data/raw/ | Real scraped problem metadata |
| `interactions.csv` | data/raw/ | Synthetic user solve history |
| `user_features.csv` | data/processed/ | Per-user skill vectors |
| `problem_features.csv` | data/processed/ | One-hot topic + difficulty features |
| `train/val/test.csv` | data/processed/ | 80/10/10 split |
| `problem_embeddings.npy` | data/embeddings/ | 384-dim sentence vectors |
| `encoders.pkl` | data/processed/ | LabelEncoders + MinMaxScaler |
| `metadata.json` | data/processed/ | Dimension counts for model init |

### ➡️ Next: Phase 2 — Model Training
- 🔁 **Collaborative Filtering** — Matrix Factorization with PyTorch
- 📐 **Content-Based** — Cosine similarity on embeddings
- 🎯 **Weak Area Detector** — Skill gap scoring
- 🔀 **Hybrid Scorer** — Weighted ensemble of all 3